In [ ]:
from tparton.m_evolution import evolve
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
from scipy.special import gamma
# From Gehrmann (https://arxiv.org/pdf/hep-ph/9512406)
n = 3000
x = np.power(10, np.linspace(np.log10(1/3000), 0, n))
x = np.concatenate(([0], x))

def A(a, b, g, rho):
    return (1 + g * a / (a + b + 1)) \
        * gamma(a) * gamma(b+1) / gamma(a + b + 1) \
        + rho * gamma(a + 0.5) * gamma(b + 1) / gamma(a + b + 1.5)

def pdf_u(x, eta_u=0.918, a_u=0.512, b_u=3.96, gamma_u=11.65, rho_u=-4.60):
    return eta_u / A(a_u, b_u, gamma_u, rho_u) * np.power(x, a_u) * np.power(1-x, b_u) * (1 + gamma_u * x + rho_u * np.sqrt(x))

def pdf_d(x, eta_d=-0.339, a_d=0.780, b_d=4.96, gamma_d=7.81, rho_d=-3.48):
    return pdf_u(x, eta_d, a_d, b_d, gamma_d, rho_d)

# Figure 2 uses degree 7 in Cohen's method, the degree of smallest error in the
# benchmark of Figure 4. Figures 1 and 3 keep the default degree 5, so they are
# produced by "Running Vogelsang.ipynb" and "Running Vogelsang-exact_alpha.ipynb".
degree = 7
diff = np.stack((x, pdf_u(x) - pdf_d(x))).T

In [ ]:
outputs = {}
for name, alpha_num in (('ana', False), ('num', True)):
    output_plus = evolve(diff, Q0_2=4, Q2=200, n_x=n, l_QCD=0.231, n_f=4, morp='plus', verbose=True, alpha_num=alpha_num, degree=degree)
    output_minus = evolve(diff, Q0_2=4, Q2=200, n_x=n, l_QCD=0.231, n_f=4, morp='minus', verbose=True, alpha_num=alpha_num, degree=degree)
    outputs[name] = (output_plus[1] - output_minus[1]) / 2
x = output_plus[0]

In [ ]:
plt.plot(x, outputs['ana'], label=r'analytical $\alpha_S$')
plt.plot(x, outputs['num'], label=r'numerical $\alpha_S$')
plt.xscale('log')
plt.xlim([2/3000, 1])
plt.ylim([0, 0.00025])
plt.legend()

In [ ]:
np.savez('vogelsang_fig2_deg7.npz', x=x, output_ana=outputs['ana'], output_num=outputs['num'], degree=degree)